In [ ]:
## IMPORTS ##
from datetime import datetime
import os
from pathlib import Path
import pandas as pd
import requests
from dotenv import load_dotenv
from openpyxl import load_workbook
from openpyxl.formatting.rule import Rule
from openpyxl.styles import (
    Alignment,
    Border,
    Font,
    PatternFill,
    Side,
)
from openpyxl.styles.differential import DifferentialStyle
from openpyxl.utils import get_column_letter

In [ ]:
## SCRIPT HEADER ###
# This section sets the directory. Loads env variables & tells the script to fetch new data or fetch
# retrieved data again. Last data would be retrieved for training purposes. 

# The notebook is located in the project directory.
PROJECT_DIR = Path.cwd()

# Load the Apify token from .env.
load_dotenv(PROJECT_DIR / ".env")

# Choose "fresh" for a new Actor search or "last" for the previous dataset.
DATA_SOURCE = "last"

# Select the current job category.
ACTIVE_SEARCH = "Financial_Analyst"

# Load search URLs from the .env file.
SEARCH_CONFIGS = {
    "Financial_Analyst": {
        "sheet_name": "Financial_Analyst",
        "start_urls": [
            os.getenv("FINANCIAL_ANALYST_URL_1"),
            os.getenv("FINANCIAL_ANALYST_URL_2"),
        ],
    },
}

active_config = SEARCH_CONFIGS[ACTIVE_SEARCH]
SHEET_NAME = active_config["sheet_name"]

if any(not url for url in active_config["start_urls"]):
    raise RuntimeError(
        f"Missing URL configuration for {ACTIVE_SEARCH} in .env."
    )

# Endpoint for the last successful Actor dataset.
LAST_DATASET_URL = (
    "https://api.apify.com/v2/actors/"
    "cheap_scraper~linkedin-job-scraper/"
    "runs/last/dataset/items"
)

# Endpoint for starting a fresh Actor run and returning its dataset.
RUN_DATASET_URL = (
    "https://api.apify.com/v2/acts/"
    "cheap_scraper~linkedin-job-scraper/"
    "run-sync-get-dataset-items"
)

In [ ]:
def get_api_token():
    """Return the Apify token or stop if it is missing."""

    token = os.getenv("APIFY_API_TOKEN")

    if not token:
        raise RuntimeError("APIFY_API_TOKEN is missing from the .env file.")

    return token

In [ ]:
## LOAD DATA THROUGH API ###

# For testing pruposes sometimes i might load last fetched data to improve/test some logic
def load_last_dataset():
    """Load jobs from the last successful Actor run."""

    headers = {
        "Authorization": f"Bearer {get_api_token()}"
    }

    parameters = {
        "status": "SUCCEEDED",
    }

    response = requests.get(
        LAST_DATASET_URL,
        headers=headers,
        params=parameters,
        timeout=60,
    )

    response.raise_for_status()

    return response.json()

# This function fetches new data from API 
def fetch_fresh_jobs(start_urls):
    """Start a new Actor search and return its job records."""

    headers = {
        "Authorization": f"Bearer {get_api_token()}"
    }

    run_input = {
        "startUrls": [{"url": url} for url in start_urls],
        "maxItems": 150,
        "saveOnlyUniqueItems": True,
    }

    response = requests.post(
        RUN_DATASET_URL,
        headers=headers,
        json=run_input,
        timeout=300,
    )

    response.raise_for_status()

    return response.json()

In [ ]:
# Load jobs using the selected data source.
# Fresh = New jobs 
# last = Previously fetched jobs
if DATA_SOURCE == "fresh":
    jobs = fetch_fresh_jobs(active_config["start_urls"])
elif DATA_SOURCE == "last":
    jobs = load_last_dataset()
else:
    raise ValueError(
        "DATA_SOURCE must be either 'fresh' or 'last'."
    )

# Record the number of jobs returned directly by the API.
total_jobs_fetched = len(jobs)

print(f"Total Jobs fetched through API: {total_jobs_fetched}")

Total Jobs fetched through API: 95


In [ ]:
# Convert JSON to Dataframe
df = pd.DataFrame(jobs)

print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

Rows: 95
Columns: 24


In [ ]:
#df

In [ ]:
## DATA CLEANING & TRANSFORMATION ## 

# Keep only the required API columns.
columns_to_keep = [
    "jobId",
    "jobTitle",
    "companyName",
    "location",
    "publishedAt",
    "sector",
    "jobUrl",
    "posterFullName",
    "posterProfileUrl",
    "companyId",
]

# Stop if the API response is missing an expected column.
missing_columns = [
    column for column in columns_to_keep
    if column not in df.columns
]

if missing_columns:
    raise KeyError(f"Missing expected columns: {missing_columns}")

# Select the required columns in the preferred order.
df = df[columns_to_keep].copy()

# Keep job IDs as text and convert publication dates to datetimes.
df["jobId"] = df["jobId"].astype("string")
df["publishedAt"] = pd.to_datetime(
    df["publishedAt"],
    errors="coerce",
    utc=True,
)

# Sort first so the newest copy is retained for duplicate job IDs.
df = (
    df.sort_values(
        "publishedAt",
        ascending=False,
        na_position="last",
    )
    .drop_duplicates(
        subset="jobId",
        keep="first",
    )
    .reset_index(drop=True)
)


In [ ]:
## Custom Columns ##
# These customs columns can be used by user to fill in info 

# Free-text notes about the job or application.
df["Comment"] = ""

# Mark with X after submitting an application.
# Color Formatting will be added if users inputs X - later in script
df["Applied?"] = ""

# Mark with X to remove and permanently exclude the job.
# Color Formatting will be added if users inputs X - later in script
df["Not Wanted"] = ""


In [ ]:
## EXCEL CONFIGURATION ## 

# Blacklist csv contain jobs which were marked by user as "Not Wanted"
# Daily log csv makes a record of the info fetched through each run of the script 
# Things such date of execution, jobs for which sector?, how many jobs fetched, how many jobs removed from excel sheet. 

EXCEL_PATH = PROJECT_DIR / "Full_time_job_listings.xlsx"
BLACKLIST_PATH = PROJECT_DIR / "not_wanted_job_ids.csv"
LOG_PATH = PROJECT_DIR / "daily_run_log.csv"

# The following columns will be shown in the final excel sheet. 
excel_columns = [
    "jobId",
    "jobTitle",
    "companyName",
    "location",
    "publishedAt",
    "Days Ago",
    "sector",
    "jobUrl",
    "posterFullName",
    "posterProfileUrl",
    "Comment",
    "Applied?",
    "Not Wanted",
]

In [ ]:
# Load Existing Records
# This section hanldes data already stored in excel sheet from previous runs 
# We load the worksheet and blacklist csv. 

# Load the existing sector worksheet when it is available.
if EXCEL_PATH.exists():
    with pd.ExcelFile(EXCEL_PATH) as workbook:
        sheet_exists = SHEET_NAME in workbook.sheet_names

    if sheet_exists:
        existing_df = pd.read_excel(
            EXCEL_PATH,
            sheet_name=SHEET_NAME,
            dtype={"jobId": "string"},
        )
    else:
        existing_df = pd.DataFrame(columns=excel_columns)
else:
    existing_df = pd.DataFrame(columns=excel_columns)

# Add columns that are missing from an older worksheet.
for column in excel_columns:
    if column not in existing_df.columns:
        existing_df[column] = ""

existing_df = existing_df[excel_columns]
existing_df["jobId"] = existing_df["jobId"].astype("string").str.strip()

# Load job IDs that were previously marked as not wanted.
if BLACKLIST_PATH.exists():
    blacklist_df = pd.read_csv(
        BLACKLIST_PATH,
        dtype={"jobId": "string"},
    )

    if "jobId" not in blacklist_df.columns:
        raise KeyError("The blacklist CSV must contain a jobId column.")

    blacklisted_ids = set(
        blacklist_df["jobId"].dropna().str.strip()
    )
else:
    blacklisted_ids = set()

blacklisted_ids.discard("")

In [ ]:
## FILTER & MERGER RECORDS ## 
# This section updates the Excel data. We add new jobs to the Excel dataframe. 
# We update the Blacklist csv with "Not Wanted" jobs and also remove them from Excel dataframe
# only completey job not previously present will be added to Excel 

# Find existing Excel rows newly marked with X.
not_wanted_mask = (
    existing_df["Not Wanted"]
    .fillna("")
    .astype("string")
    .str.strip()
    .str.casefold()
    .eq("x")
)

newly_blacklisted_ids = set(
    existing_df.loc[not_wanted_mask, "jobId"]
    .dropna()
    .str.strip()
)

newly_blacklisted_ids.discard("")
blacklisted_ids.update(newly_blacklisted_ids)

# Remove not-wanted rows and duplicate IDs from existing jobs.
existing_df = (
    existing_df.loc[~not_wanted_mask]
    .drop_duplicates(subset="jobId", keep="first")
    .copy()
)

existing_ids = set(existing_df["jobId"].dropna().str.strip())
existing_ids.discard("")

# Keep API jobs that are new and not blacklisted.
new_jobs_df = df.copy()
new_jobs_df["jobId"] = new_jobs_df["jobId"].astype("string").str.strip()

valid_id_mask = new_jobs_df["jobId"].notna() & new_jobs_df["jobId"].ne("")
excluded_ids = existing_ids | blacklisted_ids

new_jobs_df = new_jobs_df.loc[
    valid_id_mask & ~new_jobs_df["jobId"].isin(excluded_ids)
].copy()

# Exclude companyId from the Excel output.
new_jobs_df = new_jobs_df.reindex(columns=excel_columns)

# Place new and existing jobs together.
combined_df = pd.concat(
    [new_jobs_df, existing_df],
    ignore_index=True,
)

# Sort by publication date before removing the time component.
combined_df["publishedAt"] = pd.to_datetime(
    combined_df["publishedAt"],
    errors="coerce",
    utc=True,
)

combined_df = combined_df.sort_values(
    "publishedAt",
    ascending=False,
    na_position="last",
    kind="stable",
).reset_index(drop=True)

# Keep only the publication date for Excel.
combined_df["publishedAt"] = combined_df["publishedAt"].dt.date

In [ ]:
# Write Data to Excel File

if EXCEL_PATH.exists():
    with pd.ExcelWriter(
        EXCEL_PATH,
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace",
    ) as writer:
        combined_df.to_excel(
            writer,
            sheet_name=SHEET_NAME,
            index=False,
        )
else:
    combined_df.to_excel(
        EXCEL_PATH,
        sheet_name=SHEET_NAME,
        index=False,
        engine="openpyxl",
    )

# Save job IDs that must not be added again.
pd.DataFrame(
    {"jobId": sorted(blacklisted_ids)}
).to_csv(BLACKLIST_PATH, index=False)

print(f"New jobs written: {len(new_jobs_df)}")
print(f"Not-wanted jobs removed: {not_wanted_mask.sum()}")
print(f"Total jobs in worksheet: {len(combined_df)}")

New jobs written: 0
Not-wanted jobs removed: 4
Total jobs in worksheet: 91


In [ ]:
# Format Excel File
# We add format the header make it bold, add borders around the table 
# also add conditional formating in excel sheet to highlight rows market with light green for applied 
# light red for jobs not wanted. 
# Purpose is to make output more presentable to the user 

workbook = load_workbook(EXCEL_PATH)
worksheet = workbook[SHEET_NAME]

# Format the header row.
header_fill = PatternFill(
    patternType="solid",
    fgColor="FFD9D9D9",
)

for cell in worksheet[1]:
    cell.font = Font(bold=True, size=16)
    cell.fill = header_fill
    cell.alignment = Alignment(
        horizontal="center",
        vertical="center",
    )

# Add thin borders around every cell in the table.
thin_side = Side(style="thin", color="FF808080")
table_border = Border(
    left=thin_side,
    right=thin_side,
    top=thin_side,
    bottom=thin_side,
)

for row in worksheet.iter_rows(
    min_row=1,
    max_row=worksheet.max_row,
    min_col=1,
    max_col=worksheet.max_column,
):
    for cell in row:
        cell.border = table_border

# Keep the header visible and enable Excel filters.
worksheet.freeze_panes = "A2"
worksheet.auto_filter.ref = worksheet.dimensions

# Adjust widths while preventing URL columns from becoming enormous.
for column_number in range(1, worksheet.max_column + 1):
    column_letter = get_column_letter(column_number)

    longest_value = max(
        len(str(worksheet.cell(row, column_number).value or ""))
        for row in range(1, worksheet.max_row + 1)
    )

    worksheet.column_dimensions[column_letter].width = min(
        max(longest_value + 2, 12),
        60,
    )

# Highlight applied jobs in light green.
headers = {cell.value: cell.column for cell in worksheet[1]}

# Calculate Days Ago in Excel so it updates automatically.
published_column_letter = get_column_letter(headers["publishedAt"])
days_ago_column = headers["Days Ago"]

for row_number in range(2, worksheet.max_row + 1):
    days_ago_cell = worksheet.cell(row_number, days_ago_column)
    days_ago_cell.value = (
        f'=IF(${published_column_letter}{row_number}="","",'
        f'TODAY()-${published_column_letter}{row_number})'
    )
    days_ago_cell.number_format = "0"

# Keep the original URL visible and clickable.
job_url_column = headers["jobUrl"]

for row_number in range(2, worksheet.max_row + 1):
    url_cell = worksheet.cell(row_number, job_url_column)
    raw_url = str(url_cell.value or "").strip()

    if raw_url.startswith(("http://", "https://")):
        url_cell.hyperlink = raw_url
        url_cell.style = "Hyperlink"
        url_cell.alignment = Alignment(
            wrap_text=True,
            vertical="top",
        )

applied_column = headers["Applied?"]
applied_column_letter = get_column_letter(applied_column)
last_column_letter = get_column_letter(worksheet.max_column)
applied_fill = PatternFill(
    patternType="solid",
    fgColor="FFC6EFCE",
    bgColor="FFC6EFCE",
)

# Highlight the table row dynamically when Applied? contains X.
applied_rule = Rule(
    type="expression",
    dxf=DifferentialStyle(fill=applied_fill),
    formula=[f'TRIM(UPPER(${applied_column_letter}2))="X"'],
)

worksheet.conditional_formatting.add(
    f"A2:{last_column_letter}{worksheet.max_row}",
    applied_rule,
)

# Highlight not-wanted jobs in light red.
not_wanted_column = headers["Not Wanted"]
not_wanted_column_letter = get_column_letter(not_wanted_column)

not_wanted_fill = PatternFill(
    patternType="solid",
    fgColor="FFFFC7CE",
    bgColor="FFFFC7CE",
)

not_wanted_rule = Rule(
    type="expression",
    dxf=DifferentialStyle(fill=not_wanted_fill),
    formula=[f'TRIM(UPPER(${not_wanted_column_letter}2))="X"'],
)

worksheet.conditional_formatting.add(
    f"A2:{last_column_letter}{worksheet.max_row}",
    not_wanted_rule,
)

# Leave the worksheet completely editable.
worksheet.protection.sheet = False

workbook.save(EXCEL_PATH)

In [ ]:
# Daily Run Log
# Date: When was the script executed
# Sector: Which Sheet did we update? 
# total_jobs_fetched: How many jobs were given by the API 
# new_jobs_added_how many jobs were added to the excel sheet 
# how many jobs were removed from the excel sheet

# Create one log record for the current run.
run_log_row = pd.DataFrame(
    [
        {
            "date": datetime.now().astimezone().strftime("%Y-%m-%d"),
            "sector": SHEET_NAME,
            "total_jobs_fetched": len(jobs),
            "new_jobs_added": len(new_jobs_df),
            "jobs_removed": int(not_wanted_mask.sum()),
        }
    ]
)

# Load previous log records when the file already exists.
if LOG_PATH.exists():
    previous_log = pd.read_csv(LOG_PATH)
    daily_log = pd.concat(
        [previous_log, run_log_row],
        ignore_index=True,
    )
else:
    daily_log = run_log_row

# Save the updated history.
daily_log.to_csv(LOG_PATH, index=False)

daily_log.tail()

,date,sector,total_jobs_fetched,new_jobs_added,jobs_removed
0,2026-07-26,Financial_Analyst,95,95,0
1,2026-07-26,Financial_Analyst,95,0,4
